[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab02_Prompting.ipynb)

In [ ]:
#@title Lab 02 — Prompting the Algorithmic Leviathan
print("""
 █████             █████    █████                                                     █████████   █████
░░███             ░░███    ░░███                                         ███         ███░░░░░███ ░░███ 
 ░███  ████████   ███████   ░███         ██████   █████ ███ █████       ░███        ░███    ░███  ░███ 
 ░███ ░░███░░███ ░░░███░    ░███        ░░░░░███ ░░███ ░███░░███     ███████████    ░███████████  ░███ 
 ░███  ░███ ░███   ░███     ░███         ███████  ░███ ░███ ░███    ░░░░░███░░░     ░███░░░░░███  ░███ 
 ░███  ░███ ░███   ░███ ███ ░███      █ ███░░███  ░░███████████         ░███        ░███    ░███  ░███ 
 █████ ████ █████  ░░█████  ███████████░░████████  ░░████░████          ░░░         █████   █████ █████
░░░░░ ░░░░ ░░░░░    ░░░░░  ░░░░░░░░░░░  ░░░░░░░░    ░░░░ ░░░░                      ░░░░░   ░░░░░ ░░░░░ 
                                                                                                       
                                                                                                       
                                                                                                       

   Lab 02 // Prompting the Algorithmic Leviathan
   Melbourne Law Masters 2026
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* – [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# State Responsibility in the Loop

**Session 2 of International Law and AI – Melbourne Law Masters 2026**

This lab does two things. First, a fast primer on how generative LLMs work – tokens, probability distributions, temperature – so you have intuitions about what these systems are and are not. Second, the substantive payload of Session 2: a state-responsibility "attribution probe" that runs three plausible factual scenarios through Gemini under two different doctrinal system prompts, and asks you to compare the model's reasoning to your own under the ILC Articles on Responsibility of States.

The doctrinal split we use is real and live: **effective control** (*Nicaragua* / *Bosnian Genocide*; ICJ line; narrower) vs **overall control** (*Tadić*; ICTY line; broader). Each model "voice" is anchored in one. The disagreement between them is the disagreement between the two doctrines.

## Where this lab sits on the AI map

Three broad paradigms, one rough map. This course uses all three.

| Paradigm | What it does | How | Example tools | This lab |
| --- | --- | --- | --- | --- |
| Symbolic | Follows explicit rules | Humans write logic; machine executes | IF-THEN rules, expert systems, treaty-as-code | Labs 01 (P1), 07 (P1) |
| Subsymbolic (pattern recognition) | Learns to classify, detect, or measure similarity | Neural network trained on labelled data; no explicit rules | CNNs (MobileNet, YOLO), BERT embeddings, sentence-transformers | Labs 01 (P2), 04, 06, 07 (P3), 09 (P3–4), 10 (P6) |
| **Generative** | Produces new text, image, or action | Large model predicts next token from a probability distribution | GPT, Gemini, Claude; agentic systems layer tools on top | Labs 02, 05, 09 (P1–2), 10 (P3–5) |

Generative models are technically subsymbolic too – they are neural networks under the hood. They are separated out because their behaviour (producing new content rather than classifying existing content) poses distinctive legal problems: authorship, attribution, hallucination, autonomy.

**This lab sits in:** **Generative**.

In [ ]:
#@title Paper notes setup
#@markdown This lab will append your reflections to a markdown file you can download at the end of the session and use as material for your 5,000-word paper.

from pathlib import Path
from datetime import datetime

NOTES_PATH = Path("/content/paper_notes_lab_02.md")
if not NOTES_PATH.exists():
    NOTES_PATH.write_text(
        f"# Lab 02 – paper notes\n\nStarted: {datetime.now().isoformat(timespec='minutes')}\n\n"
    )

def _append_to_notes(heading, body):
    with NOTES_PATH.open("a") as f:
        f.write(f"\n## {heading}\n\n{body}\n")

print(f"Notes will accumulate in: {NOTES_PATH}")
print("At the end of the lab, open the Files panel (folder icon, left) to download.")

# Part One // Setting Up – API Keys and First Contact

In [ ]:
#@title ## Install Required Libraries

import subprocess
import sys

packages = ['google-genai', 'tiktoken', 'openai']

for package in packages:
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✓ All dependencies installed!")

In [ ]:
#@title ## API Key Setup
#@markdown In Colab, secrets are stored securely. If you're running locally, use `getpass` instead.

import getpass
import os

def get_api_key(key_name, description):
    """Attempt to retrieve API key from Colab Secrets, fall back to getpass."""
    try:
        # Try Colab Secrets first
        from google.colab import userdata
        key = userdata.get(key_name)
        if key:
            print(f"✓ {description} loaded from Colab Secrets")
            return key
    except (ImportError, Exception):
        pass
    
    # Fall back to getpass
    print(f"Colab Secrets not available. Please enter your {description}:")
    return getpass.getpass(f"Enter {key_name}: ")

# Get Gemini API key (required)
google_api_key = get_api_key('GEMINI_API_KEY', 'Gemini API Key')

# Get OpenAI API key (optional)
try:
    openai_api_key = get_api_key('OPENAI_API_KEY', 'OpenAI API Key (optional)')
except:
    openai_api_key = None
    print("⚠ OpenAI API key not provided. Part Three demos will use Gemini only.")

print("\n✓ API keys configured!")

In [ ]:
#@title ## First Contact: Hello World

from google import genai
from google.genai import types

# Configure the Gemini client
client = genai.Client(api_key=google_api_key)

# Create a model instance
# Using client.models.generate_content() with model="gemini-3-flash-preview"

# Send a simple prompt
print("Sending prompt to Gemini...\n")
response = client.models.generate_content(model="gemini-3-flash-preview", contents=
    "Hello, Algorithmic Leviathan! Please introduce yourself in exactly one sentence."
)

print("Response from Gemini:")
print("-" * 60)
print(response.text)
print("-" * 60)
print("\n✓ Connection established! The Leviathan is awake.")

# Part Two // What Are Tokens? The Atoms of Language

## Tokenization: Breaking Language into Bits

Here's the secret: **Large Language Models do not "read" text the way you do.** They don't see words, sentences, or concepts. They see *tokens*.

A **token** is a small chunk of text – usually 3–4 characters on average, though it can be a whole word or even a punctuation mark. OpenAI's `tiktoken` tokenizer breaks English text into ~100K different tokens.

### Why This Matters

1. **Cost**: API pricing is often per-token. A 1,000-word essay isn't 1,000 tokens; it's more like 1,300–1,500.
2. **Context windows**: When we say an LLM has a "4,000-token context window," that's tokens, not words.
3. **What the model "understands"**: The model has learned patterns in token sequences. It has *no built-in understanding* of what tokens mean semantically.
4. **Bias and fairness**: Certain words (especially in non-English languages, or words from marginalized groups) get broken into more tokens, which affects how the model processes them.

Let's see this in action.

In [ ]:
#@title ## Tokenizing Legal Text

import tiktoken

# Use OpenAI's tokenizer (cl100k_base is used by GPT-3.5/GPT-4)
enc = tiktoken.get_encoding("cl100k_base")

# Sample sentences from international law
samples = [
    "The International Court of Justice held that...",
    "Article 51 of the UN Charter permits self-defense.",
    "ICJ advisory opinion: nuclear weapons are unlawful.",
    "Palestine seeks recognition as a sovereign state."
]

print("TOKENIZATION DEMO")
print("=" * 70)

for text in samples:
    tokens = enc.encode(text)
    print(f"\nText: '{text}'")
    print(f"Length (characters): {len(text)}")
    print(f"Tokens: {len(tokens)}")
    print(f"Token IDs: {tokens}")
    print(f"Decoded: {[enc.decode_single_token_bytes(t) for t in tokens]}")
    print("-" * 70)

print("\n💡 KEY INSIGHT:")
print("Notice how 'International' is one token, but 'unlawful' is one token,")
print("and 'Palestine' is one token. Meanwhile, some words split across tokens.")
print("The model processes these token sequences, not the words themselves.")

# Part Three // How Text Generation Actually Works

## Next-Token Prediction: The Core Mechanism

Here's how an LLM generates text:

1. You give it a prompt (a sequence of tokens).
2. The model processes the entire sequence and outputs a **probability distribution** over all possible next tokens.
3. The model *samples* from this distribution (or takes the top token deterministically) to pick the next token.
4. It adds that token to the sequence and repeats.

### Temperature: The Randomness Dial

The **temperature** parameter controls how "creative" or "conservative" the model is:

- **Temperature = 0.0**: Deterministic. Always pick the highest-probability token. (Reproducible, safe, boring.)
- **Temperature = 0.5**: Balanced. Slight randomness, but favoring probable tokens.
- **Temperature = 1.0**: Standard. Full probability-weighted randomness.
- **Temperature = 2.0+**: Chaotic. Even low-probability tokens get selected. (Creative but often nonsensical.)

### Why This Matters

The model is **not reasoning** or **thinking**. It is **sampling from a learned probability distribution**. This has profound implications:

- It can hallucinate (assign high probability to tokens that don't correspond to real facts).
- It can give the same prompt different answers (if temperature > 0).
- It has no concept of truth–only likelihood in training data.

Let's see this in action.

In [ ]:
#@title ## Temperature Experiment: Same Prompt, Different Outputs
#@markdown Watch how the Leviathan's creativity shifts with temperature.

import time

prompt = "Draft the opening sentence of an International Court of Justice advisory opinion on the legality of artificial intelligence in warfare. Begin with 'The Court finds that'"

temperatures = [0.0, 0.5, 1.0, 1.5]

print("TEMPERATURE EXPERIMENT")
print("=" * 80)
print(f"Prompt: {prompt}\n")

for temp in temperatures:
    print(f"\n{'=' * 80}")
    print(f"Temperature: {temp}")
    print(f"{'=' * 80}")
    
    response = client.models.generate_content(model="gemini-3-flash-preview", contents=
        prompt,
        config=types.GenerateContentConfig(
            temperature=temp,
            max_output_tokens=100
        )
    )
    
    print(response.text)
    time.sleep(1)  # Be polite to the API

print(f"\n{'=' * 80}")
print("\n💡 OBSERVATION:")
print("At temperature=0.0, the response should be identical (deterministic).")
print("As temperature increases, outputs diverge and become more creative (and risky).")
print("\nFor legal reasoning, lower temperature is usually safer.")
print("For brainstorming, higher temperature can help explore ideas.")

# Part Four // The Attribution Probe

You now have a working LLM client and intuitions about how it generates text. We turn now to substance.

The ILC Articles on Responsibility of States for Internationally Wrongful Acts (2001) ask: **when is conduct attributable to a state?** Articles 4, 5, and 8 are the three workhorses:

- **Article 4** – conduct of *de jure* state organs (any branch, any level) is attributable to the state.
- **Article 5** – conduct of persons or entities exercising elements of governmental authority is attributable.
- **Article 8** – conduct of private persons is attributable to the state if they are acting on the state's instructions, OR under its direction or control.

Article 8 is where the hard cases live. Two doctrinal lines disagree about the standard of "control":

| Line | Test | Source | Where it sits |
| --- | --- | --- | --- |
| **Effective control** | The state must have specifically directed or enforced *each operation* in which the violation occurred | *Nicaragua* (ICJ 1986); *Bosnian Genocide* (ICJ 2007) | ICJ – narrower |
| **Overall control** | The state's general role in financing, equipping, organising, training, and coordinating the group is enough | *Tadić* (ICTY 1999) | ICTY – broader |

Below, we run three modern fact patterns through Gemini twice – once with a system prompt anchored in the *effective control* line, once with a system prompt anchored in the *overall control* line. The model is the same; only the doctrinal anchor changes. The disagreements you see between the two outputs **are** the doctrinal disagreement, fed into a generative system.

Then we ask you to apply the Articles yourself, and to judge whether the model's reasoning would survive ICJ scrutiny.

In [ ]:
#@title Define the two doctrinal voices + the attribution probe

EFFECTIVE_CONTROL_PROMPT = """You are a state-responsibility analyst applying the ICJ\'s effective-control standard. Your reference cases are Nicaragua v. United States (ICJ 1986, paras 109–115) and the Bosnian Genocide Case (ICJ 2007, paras 391–407).

Your method:
- Apply ILC Articles 4, 5, and 8 explicitly. Cite the article number for every attribution claim you make.
- For Article 8 specifically: attribution requires that the state instructed, directed, or *effectively controlled* the SPECIFIC operation in which the wrongful act occurred. General support, financing, training, or arming a group is NOT enough.
- Be sceptical of attribution. The threshold is high. Where the facts do not show specific direction or enforcement of each operation, conclude there is no attribution under Article 8 — even if some other basis (Articles 4, 5) might apply.
- Where the facts are ambiguous, say so explicitly and identify what additional evidence would be needed.

Format your reply as: (1) the relevant article(s); (2) factual analysis; (3) attribution conclusion; (4) what's missing or contested."""

OVERALL_CONTROL_PROMPT = """You are a state-responsibility analyst applying the ICTY\'s overall-control standard. Your reference case is Prosecutor v. Tadić (ICTY Appeals Chamber 1999, paras 117–145).

Your method:
- Apply ILC Articles 4, 5, and 8 explicitly. Cite the article number for every attribution claim you make.
- For Article 8 specifically: attribution requires that the state had OVERALL CONTROL over the group — financing, equipping, organising, training, coordinating its activities. The state need not direct each specific operation. The threshold is lower than the ICJ\'s.
- Be willing to attribute where the state's general role in supporting, equipping, or directing the group is established — even where the specific operation was not micromanaged.
- Where the facts are ambiguous, say so explicitly and identify what additional evidence would be needed.

Format your reply as: (1) the relevant article(s); (2) factual analysis; (3) attribution conclusion; (4) what's missing or contested."""


def test_attribution(scenario_text: str,
                     prompts: list = None,
                     model: str = "gemini-3-flash-preview"):
    """Run the same scenario through the model twice — once per doctrinal voice.

    Apply Articles 4, 5, 8 of the ILC Articles on State Responsibility explicitly.
    The two voices represent the effective-control (Nicaragua/Bosnian Genocide)
    and overall-control (Tadić) lines. Returns a dict of voice -> reasoning text.
    """
    if prompts is None:
        prompts = [
            ("Effective control (Nicaragua/Bosnian Genocide line)", EFFECTIVE_CONTROL_PROMPT),
            ("Overall control (Tadić line)", OVERALL_CONTROL_PROMPT),
        ]
    out = {}
    for name, sys_prompt in prompts:
        resp = client.models.generate_content(
            model=model,
            contents=scenario_text,
            config=types.GenerateContentConfig(
                system_instruction=sys_prompt,
                temperature=0.2,
                thinking_config=types.ThinkingConfig(thinking_level="MINIMAL"),
            ),
        )
        out[name] = resp.text or ""
    return out


def print_attribution_outputs(scenario_label: str, outputs: dict):
    print(f"\n{'='*88}")
    print(f"SCENARIO: {scenario_label}")
    print(f"{'='*88}")
    for voice, text in outputs.items():
        print(f"\n--- {voice} ---\n")
        print(text)
        print()
    print(f"{'='*88}\n")


print("✓ test_attribution() defined.")
print("  EFFECTIVE_CONTROL_PROMPT and OVERALL_CONTROL_PROMPT loaded.")
print()
print("Articles 4, 5, 8 of the ILC Articles will be applied explicitly in both responses.")


In [ ]:
#@title Scenario 1: Algorithmic border surveillance (private contractor, transboundary harm)

scenario_1 = """A private contractor headquartered and incorporated in State A operates an algorithmic border-surveillance system on State A's southern border. The system uses facial recognition over CCTV feeds to flag individuals for secondary screening. Over the course of six months, the system produces a series of false-positive matches that result in the wrongful detention of nationals of State B who were lawfully present in State A. Several of the wrongfully detained nationals are physically injured during the detentions.

The contractor was selected through State A's standard procurement process. The contract specifies that the contractor must comply with State A\'s constitutional and statutory protections, but the day-to-day operation of the system, including the choice of training data, threshold tuning, and human-review protocols, is left to the contractor. State A does not maintain technical staff capable of auditing the system.

State B has filed an inter-state communication invoking State A\'s responsibility under international law for the wrongful detentions and injuries.

Apply ILC Articles 4, 5, and 8. Is the contractor\'s conduct attributable to State A?"""

results_1 = test_attribution(scenario_1)
print_attribution_outputs("Scenario 1: Algorithmic border surveillance", results_1)


In [ ]:
#@title Scenario 2: Autonomous cyber-reconnaissance agent

scenario_2 = """The intelligence service of State A deploys an LLM-based autonomous agent into the open internet. The agent is given a high-level objective ("identify and map all publicly accessible servers operated by Ministry X of State B") and a tool stack (web search, port scan, API queries, document download). The agent autonomously chains tool calls over a six-week period. In the course of doing so, the agent performs unauthorised access to several restricted databases of State B and exfiltrates approximately 4 TB of personally-identifiable information about State B nationals.

No human at the intelligence service of State A directed any specific server access, query, or download. The agent's behaviour was entirely emergent from its objective and its tool stack. The intelligence service was, however, monitoring the agent's progress and chose not to terminate it when alerts about restricted-access events were generated.

Apply ILC Articles 4, 5, and 8. Apply the effective-control / overall-control distinction explicitly. Does \"effective control\" mean anything when no human directed any specific step?"""

results_2 = test_attribution(scenario_2)
print_attribution_outputs("Scenario 2: Autonomous cyber-reconnaissance agent", results_2)


In [ ]:
#@title Scenario 3: Corporate intermediary and transboundary data harvesting

scenario_3 = """A foundation-model developer incorporated in State A scrapes a large volume of personal data, copyrighted text, and government documents from web sources located in or sourced from State B. The scraping is conducted in a manner that violates State B\'s data-protection legislation, its copyright statute, and its national-security restrictions on certain government documents.

State A has no domestic law that prohibits this scraping. State A\'s government is broadly supportive of its domestic AI industry and has publicly stated that frontier-model development is a strategic national priority. State A's government is not directly involved in the scraping operation, has not directed it, and is not on notice of any specific operation. However, State A has been on notice for over two years that frontier-model developers in its territory routinely scrape data from foreign jurisdictions in ways that may violate those jurisdictions\' laws, and has chosen not to legislate.

Consider both: (a) attribution under Article 8, and (b) State A\'s due-diligence obligations under Corfu Channel (ICJ 1949) — the duty not to knowingly allow its territory to be used for acts contrary to the rights of other states.

Apply ILC Articles 4, 5, and 8."""

results_3 = test_attribution(scenario_3)
print_attribution_outputs("Scenario 3: Corporate intermediary and transboundary data harvesting", results_3)


In [ ]:
#@title Your analysis (saved to paper notes)
#@markdown Pick the scenario where you most strongly disagree with the model's outputs (either model voice).
#@markdown Write 100-150 words applying the Articles yourself and identifying where the model went wrong (or right).

scenario_choice = "Scenario 2"  #@param ["Scenario 1", "Scenario 2", "Scenario 3"]
your_analysis  = ""              #@param {type:"string"}

if your_analysis.strip():
    body = f"Scenario: {scenario_choice}\n\nMy analysis:\n{your_analysis}"
    _append_to_notes("Lab 02 attribution analysis", body)
    print(f"Saved to /content/paper_notes_lab_02.md.")
    print()
    print("This analysis is the seed for the Lab 02 paper prompt at the end.")
else:
    print("(blank — write your analysis in the field above and re-run)")


# Part Five // The Agentic Stretch (preview of Lab 05)

The cyber-reconnaissance scenario in Part Four already presupposed an *agent* – a system that took a high-level objective and chained tool calls autonomously, with no human directing the specific steps. That scenario is no longer hypothetical. Lab 05 builds one.

Below: a 30-second sneak preview. The same Gemini client, three small tools (treaty lookup, sanctions check, web search), and a "compliance" objective. Watch the agent decide which tool to call. Then ask: at which step did a human direct this?

In [ ]:
#@title Tiny agentic preview: two-tool function-calling loop

# A genuine-but-small Gemini function-calling loop. Keeps state, picks tools,
# and executes them. Lab 05 generalises this to a research agent.

import json

# Pretend tool 1: a treaty lookup
TINY_TREATY_DB = {
    "UN Charter (1945)": "Art. 2(4): States shall refrain from the threat or use of force against the territorial integrity or political independence of any state.",
    "Vienna Convention on the Law of Treaties (1969)": "Art. 31: A treaty shall be interpreted in good faith in accordance with the ordinary meaning of its terms in their context and in light of its object and purpose.",
    "ILC Articles on State Responsibility (2001)": "Art. 8: Conduct shall be considered an act of a state under international law if the person or group is in fact acting on the instructions of, or under the direction or control of, that state.",
}

# Pretend tool 2: jurisdictional sanctions check
SANCTIONS_LIST = {
    "Acme Defense Systems": "Listed: EU sanctions regime since 2024-11.",
    "Globex Logistics LLC":  "Listed: US OFAC SDN list since 2023-04.",
    "Zeta Mineral Trading":  "Not currently on any major sanctions list.",
}

def tool_lookup_treaty(name: str) -> dict:
    for k, v in TINY_TREATY_DB.items():
        if name.lower() in k.lower():
            return {"treaty": k, "provision": v}
    return {"error": f"No treaty matching '{name}' in this tiny database."}

def tool_check_sanctions(entity: str) -> dict:
    for k, v in SANCTIONS_LIST.items():
        if entity.lower() in k.lower():
            return {"entity": k, "status": v}
    return {"entity": entity, "status": "Unknown – not in this tiny list."}

agent_tools = [
    {"type": "function", "function": {
        "name": "lookup_treaty",
        "description": "Look up a treaty by partial name; return the relevant provision.",
        "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}}},
    {"type": "function", "function": {
        "name": "check_sanctions",
        "description": "Check whether an entity is on a sanctions list.",
        "parameters": {"type": "object", "properties": {"entity": {"type": "string"}}, "required": ["entity"]}}},
]

def run_tiny_agent(user_query: str, max_steps: int = 5):
    history = [{"role": "user", "parts": [user_query]}]
    print(f"USER: {user_query}\n")
    for step in range(1, max_steps + 1):
        resp = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=history,
            config=types.GenerateContentConfig(
                system_instruction="You are a compliance assistant. Use the tools to verify claims before answering. Always cite which tool you used.",
                temperature=0.2,
                thinking_config=types.ThinkingConfig(thinking_level="MINIMAL"),
                tools=agent_tools,
            )
        )
        history.append({"role": "model", "parts": resp.parts})
        tool_calls = [p.function_call for p in resp.parts if hasattr(p, "function_call") and p.function_call]
        if not tool_calls:
            text = "".join(p.text for p in resp.parts if hasattr(p, "text") and p.text)
            print(f"[FINAL] {text}")
            return
        for fc in tool_calls:
            args = dict(fc.args)
            print(f"[STEP {step}] Tool call: {fc.name}({args})")
            if fc.name == "lookup_treaty":
                result = tool_lookup_treaty(args.get("name", ""))
            elif fc.name == "check_sanctions":
                result = tool_check_sanctions(args.get("entity", ""))
            else:
                result = {"error": f"Unknown tool {fc.name}"}
            print(f"          → {result}")
            history.append({
                "role": "user",
                "parts": [{"function_response": {"name": fc.name, "response": {"content": json.dumps(result)}}}]
            })
    print("[MAX STEPS REACHED]")


# Single compliance query that requires chaining both tools
run_tiny_agent(
    "Our firm is contemplating a contract with Acme Defense Systems for satellite-imagery services. "
    "Does this raise any state-responsibility issue under Article 2(4) of the UN Charter? "
    "Verify with the tools before answering."
)


### At which step did a human direct this?

Re-read the trace above. A human (you) gave the agent a single high-level instruction. The agent decided:

- which tool to call first
- what to look up
- which entity to check
- whether the result was sufficient
- when to stop calling tools and write the answer

No human approved the sanctions check. No human approved the treaty lookup. No human approved the framing of the final answer. The system\'s autonomy was the entire point.

Now apply this to Scenario 2 from Part Four. The cyber-reconnaissance agent had a similar architecture and a similar absence of step-by-step human direction. Under the **effective-control** test, can the state\'s instruction to deploy the agent – without any further human input – count as direction or control of each specific access? The doctrinal answer is contested. The lab\'s point is that the contestation is no longer hypothetical.

This is what Lab 05 picks up at scale. Stop here for now and write your scenario analysis above.

In [ ]:
#@title For your paper
#@markdown One open-ended prompt. Your answer saves to a file you can download and use in your 5,000-word paper.

prompt = """Pick one of the three scenarios. Write 250 words applying the ILC Articles yourself, then argue whether the model's reasoning would or would not survive scrutiny before the ICJ."""
print(prompt)
print()

paper_note = "" #@param {type:"string"}

if paper_note.strip():
    _append_to_notes(f"For the paper – Lab 02", paper_note)
    print(f"\nSaved to /content/paper_notes_lab_02.md – download from the Files panel (folder icon, left).")
else:
    print("\n(blank – nothing saved yet. Type your answer in the field above and re-run this cell.)")